## Adaptation of Hunstville Ignition Energy Paper for Turbopump TCA

In [91]:
import cantera as ct
import CoolProp as cp
import numpy as np
import pandas as pd

In [92]:
PSI2PA = 6894.76 # psi to pa
R2K = 0.555556 # Rankine to Kelvin
BTU2J = 1055.06 # BTU to J
LBM2KG = 0.453592 # lbm to kg

In [93]:
# Torch
fuel_torch = "hydrogen"
ox_torch = "oxygen"
mech_torch = "h2_sandiego.yaml" # torch reaction mechanism
OF_torch = 40

# MCA
fuel = "n-Dodecane"
ox = "oxygen"
mech_MCA = "A2NTC_skeletal.yaml" # MCA reaction mechanism
OF_MCA = 2.2
mdot_main = 20 * LBM2KG


T1_f = 285
T2_f = 500
T1_ox = 90
T2_ox = 500
Pc = 500 * PSI2PA


### Find Change in enthalpy required to reach Autoigniton Temperature for MCA Propellants

In [94]:
h1_ox = cp.CoolProp.PropsSI("H", "T", T1_ox, "P", Pc, ox)
h2_ox = cp.CoolProp.PropsSI("H", "T", T2_ox, "P", Pc, ox)
h1_f = cp.CoolProp.PropsSI("H", "T", T1_f, "P", Pc, fuel)
h2_f = cp.CoolProp.PropsSI("H", "T", T2_f, "P", Pc, fuel)

dh_f = h2_f - h1_f
dh_ox = h2_ox - h1_ox

print(dh_f* 10**-6, dh_ox* 10**-6)

0.550902211490163 0.5908534482069829


In [95]:
dh_mix = (1/(OF_MCA+1)) * dh_f + (OF_MCA/(OF_MCA+1)) * dh_ox # J/kg
print(f"Change in enthalpy: {dh_mix * 10**-6:0.3f} [MJ/kg]")

Change in enthalpy: 0.578 [MJ/kg]


In [96]:
# function to get masss averaged chemical potential of mixture 

def mass_avg_chem_pot(mixture):
    mu = mixture.chemical_potentials      # J/kmol
    W  = mixture.molecular_weights        # kg/kmol
    Y  = mixture.Y                        # mass fractions
    return np.dot(Y, mu / W)              # J/kg mixture

In [97]:
# find difference in chemical potentials in torch reaction to find energy released from reaction
torch_rxn = ct.Solution(mech_torch)
torch_rxn.TPY = 298, ct.one_atm, {"O2": OF_torch/(OF_torch+1), "H2":(1/(OF_torch+1)) } 

# get starting chemical potentials, equillibriate reaction and find difference
pre_rxn = mass_avg_chem_pot(torch_rxn)
torch_rxn.equilibrate("HP")
post_rxn = mass_avg_chem_pot(torch_rxn)

print(f"Torch Flame Temp: {torch_rxn.T:0.1f}")
combustion_energy_torch = np.abs(post_rxn - pre_rxn)
print(f"{combustion_energy_torch* 10**-6:0.3f}")

Torch Flame Temp: 2283.0
20.997


In [98]:
# same process for MCA Combustion energy
# find difference in chemical potentials in torch reaction to find energy released from reaction
MCA_rxn = ct.Solution(mech_MCA)
MCA_rxn.TPY = T2_f, Pc, {"O2": OF_MCA/(OF_MCA+1), "POSF10325":(1/(OF_MCA+1)) } 

# get starting chemical potentials, equillibriate reaction and find difference
pre_rxn = mass_avg_chem_pot(MCA_rxn)
MCA_rxn.equilibrate("HP")
print(MCA_rxn.T)
post_rxn = mass_avg_chem_pot(MCA_rxn)

combustion_energy_MCA = np.abs(post_rxn - pre_rxn)
print(f"{combustion_energy_MCA* 10**-6:0.3f}")

3528.088053292425
40.114


In [99]:
# Compute percent of MCA that needs to ignite for chain reaction
x = dh_mix/combustion_energy_MCA
P_ig = mdot_main * x * dh_mix
print(x*100, P_ig*10**-3)

1.4418198638985504 75.65038775348499


In [103]:
mdot_torch = P_ig / combustion_energy_torch
print(f"Minimum Igniter mdot:  {mdot_torch:0.5f} [kg/s]")

print(f"(FOS 10x)Igniter mdot: {10*mdot_torch:0.5f} [kg/s]")


Minimum Igniter mdot:  0.00360 [kg/s]
(FOS 10x)Igniter mdot: 0.03603 [kg/s]


### Tabled for now: Injecting pilot flame into a wsr
#### -> can be either (likely both) torch combustion products -> torch ignition or torch combustion products -> MCA 
#### -> may be able to incorporate flame kernel radius into this